# Гармонизированные графики финальных оценок

## Назначение

готовая к публикации визуализация уже посчитанных оценок: событийная модель, постэффекты с CI, условная воронка, ATT(g,t) — через `last_mile.plot_style`, без переоценки моделей.

## Входные данные

CSV из `outputs/final/` и `outputs/empirical/` (сводка постоценок, коэффициенты событийной модели, speed/differences при наличии).

## Результаты

PDF/PNG в `figures/empirical/`; десять рисунков, используемых в `thesis/main.tex`.

## Статус

**Вспомогательный** — ноутбук не является источником эконометрических результатов; он только стилизует уже посчитанные CSV. Источник чисел — `final_empirical_recalculation.ipynb` и `outputs/final/`.


In [1]:
from __future__ import annotations

from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# PROJECT_ROOT: notebooks/ → parents[0]; from a notebook file path, project root is parents[1]
cwd = Path.cwd().resolve()
if cwd.name.lower() in {"notebooks", "notebook"}:
    PROJECT_ROOT = cwd.parents[0]
elif (cwd / "last_mile").exists():
    PROJECT_ROOT = cwd
else:
    PROJECT_ROOT = cwd.parents[0]

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

FIG_DIR = PROJECT_ROOT / "figures" / "empirical"
FIG_COND = FIG_DIR / "conditional"
OUT_FINAL = PROJECT_ROOT / "outputs" / "final"
OUT_EMP = PROJECT_ROOT / "outputs" / "empirical"

FIG_DIR.mkdir(parents=True, exist_ok=True)
FIG_COND.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("FIG_DIR:", FIG_DIR)
print("OUT_FINAL exists:", OUT_FINAL.exists())
print("OUT_EMP exists:", OUT_EMP.exists())

PROJECT_ROOT: <resolved relative to repository root>
FIG_DIR: <under repository root>
OUT_FINAL exists: True
OUT_EMP exists: True


In [2]:
from last_mile.plot_style import (
    PALETTE as SHARED_PALETTE, TYPE_LABELS, TYPE_ORDER,
    save_figure as save_vector_figure, with_plot_style,
)

PALETTE = {
    **SHARED_PALETTE,
    "navy": SHARED_PALETTE["region_and_workmode"],
    "blue_dark": SHARED_PALETTE["region_and_workmode"],
    "blue": SHARED_PALETTE["region_only"],
    "orange": SHARED_PALETTE["workmode_only"],
    "gray_dark": SHARED_PALETTE["text"],
    "gray": SHARED_PALETTE["text_muted"],
    "gray_light": SHARED_PALETTE["grid"],
    "gray_grid": SHARED_PALETTE["grid"],
}
TYPE_COLORS = {
    "region_and_workmode": PALETTE["blue_dark"],
    "region_only": PALETTE["blue"],
    "workmode_only": PALETTE["orange"],
}

OUTCOME_LABELS = {
    "sch_flg": "Назначение встречи",
    "meet_flg": "Состоявшаяся встреча",
    "success_flg": "Успешная встреча",
    "utlz_within_25": "Утилизация в течение 25 дней",
    "t_available": "Время до первого доступного интервала",
}

OUTCOMES_MAIN = ["sch_flg", "meet_flg", "success_flg", "utlz_within_25"]

def save_figure(fig: plt.Figure, stem: str, fig_dir: Path = FIG_DIR) -> None:
    fig_dir.mkdir(parents=True, exist_ok=True)
    pdf_path = fig_dir / f"{stem}.pdf"
    save_vector_figure(fig, pdf_path, preview_png=True, preview_dpi=240)
    print(f"saved: {pdf_path.relative_to(PROJECT_ROOT)}")


def canonicalize_outcome(name: str) -> str:
    """Единое имя исхода: utlz_flg → utlz_within_25."""
    if name in {"utlz_flg", "utlz", "utilization"}:
        return "utlz_within_25"
    return str(name)

## Загрузка финальных CSV

Приоритет для постэффектов конверсий:
1. `outputs/final/main_event_study_post_summary.csv`
2. резервный вариант: `outputs/empirical/main_summary.csv`

Коэффициенты событийной модели (если есть):
- `outputs/final/event_study_coefficients.csv`
- поддержка недель: `outputs/empirical/event_week_support.csv`

Скорость и ATT(g,t):
- `outputs/empirical/speed_did_summary.csv`
- `outputs/empirical/differences_event_summary.csv` или `differences_post_average_summary.csv`

In [3]:
def first_existing(*paths: Path) -> Path | None:
    for p in paths:
        if p is not None and p.exists():
            return p
    return None


def load_post_summary() -> tuple[pd.DataFrame, Path]:
    path = first_existing(
        OUT_FINAL / "main_event_study_post_summary.csv",
        OUT_EMP / "main_summary.csv",
    )
    if path is None:
        raise FileNotFoundError(
            "Не найдены main_event_study_post_summary.csv и main_summary.csv. "
            "Сначала выполните final_empirical_recalculation / build_final_tables."
        )
    df = pd.read_csv(path)
    # нормализация имён колонок из main_summary
    rename = {}
    if "avg_post_effect_pp" in df.columns and "estimate_pp" not in df.columns:
        rename["avg_post_effect_pp"] = "estimate_pp"
    if "avg_post_effect_pp_weighted" in df.columns and "estimate_pp" not in df.columns:
        rename["avg_post_effect_pp_weighted"] = "estimate_pp"
    if "avg_post_se_pp" in df.columns and "std_error_pp" not in df.columns:
        rename["avg_post_se_pp"] = "std_error_pp"
    if "avg_post_ci_low_pp" in df.columns and "ci_lower_pp" not in df.columns:
        rename["avg_post_ci_low_pp"] = "ci_lower_pp"
    if "avg_post_ci_high_pp" in df.columns and "ci_upper_pp" not in df.columns:
        rename["avg_post_ci_high_pp"] = "ci_upper_pp"
    df = df.rename(columns=rename)
    if "outcome" in df.columns:
        df["outcome"] = df["outcome"].map(canonicalize_outcome)
    return df, path


post_summary, post_path = load_post_summary()
print("Источник сводки постоценок:", post_path.relative_to(PROJECT_ROOT))
display(post_summary.head(20))

speed_path = first_existing(
    OUT_FINAL / "speed_did_summary.csv",
    OUT_EMP / "speed_did_summary.csv",
)
speed_summary = pd.read_csv(speed_path) if speed_path else pd.DataFrame()
if speed_path:
    print("Speed source:", speed_path.relative_to(PROJECT_ROOT))
    display(speed_summary)

coef_path = first_existing(
    OUT_FINAL / "event_study_coefficients.csv",
)
coefs = pd.read_csv(coef_path) if coef_path else pd.DataFrame()
if coef_path:
    print("Coefficients source:", coef_path.relative_to(PROJECT_ROOT))
    if "outcome" in coefs.columns:
        coefs["outcome"] = coefs["outcome"].map(canonicalize_outcome)
    display(coefs.head(20))
else:
    print("Weekly coefficient CSV not found.")

diff_event_path = first_existing(
    OUT_EMP / "differences_event_summary.csv",
    OUT_FINAL / "differences_event_summary.csv",
)
diff_post_path = first_existing(
    OUT_EMP / "differences_post_average_summary.csv",
    OUT_FINAL / "differences_post_average_summary.csv",
)
diff_event = pd.read_csv(diff_event_path) if diff_event_path else pd.DataFrame()
diff_post = pd.read_csv(diff_post_path) if diff_post_path else pd.DataFrame()
if diff_event_path:
    print("Differences event source:", diff_event_path.relative_to(PROJECT_ROOT))
    if "outcome" in diff_event.columns:
        diff_event["outcome"] = diff_event["outcome"].map(canonicalize_outcome)
if diff_post_path:
    print("Differences post source:", diff_post_path.relative_to(PROJECT_ROOT))
    if "outcome" in diff_post.columns:
        diff_post["outcome"] = diff_post["outcome"].map(canonicalize_outcome)

Источник сводки постоценок: outputs\final\main_event_study_post_summary.csv


           outcome          change_type  estimate  estimate_pp  std_error  ci_lower  ci_upper  p_value   n_obs  n_hexagons  n_treated_hexagons  \
0          sch_flg  region_and_workmode    0.0093       0.9346     0.0149   -0.0198    0.0385   0.5303  102301        6160                2232   
1          sch_flg          region_only   -0.0200      -1.9955     0.0175   -0.0542    0.0143   0.2540   90710        4458                 530   
2          sch_flg        workmode_only   -0.0026      -0.2625     0.0114   -0.0250    0.0198   0.8182  116092        5676                1748   
3         meet_flg  region_and_workmode   -0.0097      -0.9733     0.0217   -0.0522    0.0327   0.6531  102247        6150                2222   
4         meet_flg          region_only    0.0198       1.9783     0.0255   -0.0303    0.0698   0.4384   90710        4458                 530   
5         meet_flg        workmode_only    0.0043       0.4334     0.0163   -0.0275    0.0362   0.7898  113234        5553  

Speed source: outputs\empirical\speed_did_summary.csv


                Тип изменения          change_type  treated-гексов  treated-заявок  pretrend_pval  avg_post_effect_days_unweighted  \
0      Смена региона и режима  region_and_workmode            2228           52905         0.0001                           1.5119   
1        Только смена региона          region_only             530           30387         0.0000                           0.4601   
2  Только смена режима работы        workmode_only            1747          104516         0.0112                          -0.1998   

   avg_post_effect_days_weighted  avg_post_effect_days  avg_post_se_days  avg_post_ci_low_days  avg_post_ci_high_days  avg_post_effect_hours_unweighted  \
0                         1.1813                1.1813            0.1204                0.9453                 1.4173                           36.2858   
1                         0.1136                0.1136            0.0829               -0.0489                 0.2760                           11.0422  

Coefficients source: outputs\final\event_study_coefficients.csv


    outcome          change_type  event_week    coef     se  ci_low  ci_high   pval  estimated  baseline_week  ci_level cluster_level  \
0   sch_flg  region_and_workmode          -8  0.0260 0.0141 -0.0016   0.0537 0.0645       True             -1    0.9500           hex   
1   sch_flg  region_and_workmode          -7  0.0265 0.0181 -0.0090   0.0621 0.1436       True             -1    0.9500           hex   
2   sch_flg  region_and_workmode          -6  0.0190 0.0186 -0.0175   0.0555 0.3075       True             -1    0.9500           hex   
3   sch_flg  region_and_workmode          -5  0.0145 0.0189 -0.0226   0.0515 0.4433       True             -1    0.9500           hex   
4   sch_flg  region_and_workmode          -4  0.0445 0.0188  0.0077   0.0813 0.0179       True             -1    0.9500           hex   
5   sch_flg  region_and_workmode          -3  0.0114 0.0184 -0.0246   0.0474 0.5350       True             -1    0.9500           hex   
6   sch_flg  region_and_workmode         

Differences event source: outputs\empirical\differences_event_summary.csv
Differences post source: outputs\empirical\differences_post_average_summary.csv


## Примечание об событийная модель коэффициентах

Недельные коэффициенты с доверительными интервалами формируются при полном прогоне `final_empirical_recalculation.ipynb` (`outputs/final/event_study_coefficients.csv`). При их отсутствии ноутбук строит столбчатые диаграммы постэффектов с CI из `main_event_study_post_summary.csv`.

In [4]:
def has_weekly_coefs(df: pd.DataFrame) -> bool:
    if df is None or df.empty:
        return False
    week_col = next(
        (c for c in ["event_week", "rel_week", "relative_week", "week"] if c in df.columns),
        None,
    )
    coef_col = next((c for c in ["coef", "estimate", "coefficient"] if c in df.columns), None)
    lo_col = next((c for c in ["ci_low", "ci_lower", "lower"] if c in df.columns), None)
    hi_col = next((c for c in ["ci_high", "ci_upper", "upper"] if c in df.columns), None)
    return all(x is not None for x in [week_col, coef_col, lo_col, hi_col])


WEEKLY_AVAILABLE = has_weekly_coefs(coefs)
if WEEKLY_AVAILABLE:
    display(Markdown(
        "**Недельные коэффициенты найдены** — строим event-study графики с CI."
    ))
else:
    display(Markdown(
        "**Недельные коэффициенты отсутствуют.** "
        "Строим столбчатые диаграммы постэффектов из сводного CSV. "
        "Чтобы восстановить `event_study_*` фигуры, повторно выполните "
        "`final_empirical_recalculation.ipynb` и экспортируйте "
        "`outputs/final/event_study_coefficients.csv`."
    ))
print("WEEKLY_AVAILABLE =", WEEKLY_AVAILABLE)

**Недельные коэффициенты найдены** — строим event-study графики с CI.

WEEKLY_AVAILABLE = True


In [5]:
def normalize_coefs(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    rename = {}
    for src, dst in [
        ("rel_week", "event_week"),
        ("relative_week", "event_week"),
        ("week", "event_week"),
        ("estimate", "coef"),
        ("coefficient", "coef"),
        ("ci_lower", "ci_low"),
        ("lower", "ci_low"),
        ("ci_upper", "ci_high"),
        ("upper", "ci_high"),
    ]:
        if src in out.columns and dst not in out.columns:
            rename[src] = dst
    out = out.rename(columns=rename)
    if "outcome" in out.columns:
        out["outcome"] = out["outcome"].map(canonicalize_outcome)
    return out


@with_plot_style
def plot_event_study_from_coefs(outcome: str, coef_df: pd.DataFrame, stem: str) -> None:
    dd = normalize_coefs(coef_df)
    sub = dd[dd["outcome"] == outcome].copy() if "outcome" in dd.columns else dd.copy()
    if sub.empty:
        print(f"[skip] no coefs for {outcome}")
        return

    scale = 100.0 if outcome != "t_available" else 1.0
    ylab = "Коэффициент, п.п." if outcome != "t_available" else "Изменение t_available, дней"

    fig, axes = plt.subplots(1, len(TYPE_ORDER), figsize=(13.5, 4.1), sharey=True)
    for ax, ctype in zip(axes, TYPE_ORDER):
        part = sub[sub["change_type"] == ctype].dropna(subset=["coef", "ci_low", "ci_high"])
        part = part.sort_values("event_week")
        ax.axvspan(-0.5, 8.3, color=PALETTE["post"], zorder=0)
        ax.axhline(0, color=PALETTE["gray_dark"], linewidth=0.85, zorder=1)
        ax.axvline(-0.5, color=PALETTE["gray_dark"], linestyle=(0, (3, 2)), linewidth=0.9, zorder=1)
        if not part.empty:
            x = part["event_week"].to_numpy()
            y = part["coef"].to_numpy() * scale
            yerr = np.vstack([
                y - part["ci_low"].to_numpy() * scale,
                part["ci_high"].to_numpy() * scale - y,
            ])
            ax.errorbar(
                x, y, yerr=yerr, fmt="o-", color=TYPE_COLORS[ctype],
                linewidth=1.6, markersize=4.5, capsize=3, zorder=3,
                markerfacecolor=PALETTE["paper"],
            )
        ax.set_title(TYPE_LABELS[ctype])
        ax.set_xlabel("Неделя относительно изменения")
        ax.grid(True, axis="y", alpha=0.35)
    axes[0].set_ylabel(ylab)
    fig.suptitle(
        f"Event-study DiD: {OUTCOME_LABELS.get(outcome, outcome)}",
        y=1.02, fontsize=13, color=PALETTE["gray_dark"],
    )
    fig.tight_layout()
    save_figure(fig, stem)
    plt.show()


@with_plot_style
def plot_post_bar_chart(
    summary: pd.DataFrame,
    *,
    outcomes: list[str],
    estimate_col: str,
    lo_col: str | None,
    hi_col: str | None,
    stem: str,
    title: str,
    ylabel: str,
    fig_dir: Path = FIG_DIR,
) -> None:
    df = summary.copy()
    if "outcome" in df.columns:
        df["outcome"] = df["outcome"].map(canonicalize_outcome)
        df = df[df["outcome"].isin(outcomes)]
    if df.empty or estimate_col not in df.columns:
        print(f"[skip bars] {stem}: empty or missing {estimate_col}")
        return

    n_out = len(outcomes)
    fig, axes = plt.subplots(1, n_out, figsize=(3.6 * n_out, 4.2), sharey=False)
    if n_out == 1:
        axes = [axes]

    for ax, outcome in zip(axes, outcomes):
        part = df[df["outcome"] == outcome] if "outcome" in df.columns else df
        xs, ys, yerr_lo, yerr_hi, colors = [], [], [], [], []
        for ctype in TYPE_ORDER:
            row = part[part["change_type"] == ctype]
            if row.empty:
                continue
            r = row.iloc[0]
            est = float(r[estimate_col]) if pd.notna(r[estimate_col]) else np.nan
            if pd.isna(est):
                continue
            xs.append(TYPE_LABELS[ctype].replace(" ", "\n"))
            ys.append(est)
            colors.append(TYPE_COLORS[ctype])
            if lo_col and hi_col and lo_col in r.index and hi_col in r.index:
                lo = float(r[lo_col]) if pd.notna(r[lo_col]) else est
                hi = float(r[hi_col]) if pd.notna(r[hi_col]) else est
                yerr_lo.append(est - lo)
                yerr_hi.append(hi - est)
            else:
                yerr_lo.append(0.0)
                yerr_hi.append(0.0)

        if not xs:
            ax.set_title(OUTCOME_LABELS.get(outcome, outcome))
            ax.text(0.5, 0.5, "нет данных", ha="center", va="center", transform=ax.transAxes)
            continue

        yerr = np.vstack([yerr_lo, yerr_hi]) if any(yerr_lo) or any(yerr_hi) else None
        ax.bar(xs, ys, color=colors, edgecolor=PALETTE["paper"], width=0.72, zorder=3)
        if yerr is not None:
            ax.errorbar(xs, ys, yerr=yerr, fmt="none", ecolor=PALETTE["gray_dark"],
                        elinewidth=1.1, capsize=4, zorder=4)
        ax.axhline(0, color=PALETTE["gray_dark"], linewidth=0.85, zorder=2)
        ax.set_title(OUTCOME_LABELS.get(outcome, outcome))
        ax.grid(True, axis="y", alpha=0.35, zorder=0)
        ax.tick_params(axis="x", labelsize=7.5)

    axes[0].set_ylabel(ylabel)
    fig.suptitle(title, y=1.02, fontsize=13, color=PALETTE["gray_dark"])
    fig.tight_layout()
    save_figure(fig, stem, fig_dir=fig_dir)
    plt.show()

## Основные графики конверсий и скорости

При наличии недельных коэффициентов — событийная модель; иначе — столбцы постэффектов с CI.

In [6]:
STEM_MAP = {
    "sch_flg": "event_study_sch_by_type",
    "meet_flg": "event_study_meet_by_type",
    "success_flg": "event_study_success_by_type",
    "utlz_within_25": "event_study_utlz_by_type",
    "t_available": "event_study_t_available_by_type",
}

if WEEKLY_AVAILABLE:
    for outcome in OUTCOMES_MAIN:
        plot_event_study_from_coefs(outcome, coefs, STEM_MAP[outcome])
    if "outcome" in coefs.columns and (coefs["outcome"] == "t_available").any():
        plot_event_study_from_coefs("t_available", coefs, STEM_MAP["t_available"])
    elif not speed_summary.empty:
        # speed bars as fallback for t_available when only summary exists
        sp = speed_summary.copy()
        est_col = next(
            (c for c in ["avg_post_effect_days", "avg_weighted", "avg_post_effect_days_weighted"] if c in sp.columns),
            None,
        )
        if est_col:
            sp["outcome"] = "t_available"
            plot_post_bar_chart(
                sp,
                outcomes=["t_available"],
                estimate_col=est_col,
                lo_col=None,
                hi_col=None,
                stem="post_effect_t_available_by_type",
                title="Средний post-эффект: t_available (без недельных CI)",
                ylabel="Дни",
            )
else:
    # CI bar charts from post summary
    est = "estimate_pp" if "estimate_pp" in post_summary.columns else "avg_post_effect_pp"
    lo = "ci_lower_pp" if "ci_lower_pp" in post_summary.columns else (
        "avg_post_ci_low_pp" if "avg_post_ci_low_pp" in post_summary.columns else None
    )
    hi = "ci_upper_pp" if "ci_upper_pp" in post_summary.columns else (
        "avg_post_ci_high_pp" if "avg_post_ci_high_pp" in post_summary.columns else None
    )
    plot_post_bar_chart(
        post_summary,
        outcomes=OUTCOMES_MAIN,
        estimate_col=est,
        lo_col=lo,
        hi_col=hi,
        stem="post_effect_conversions_by_type",
        title="Средний post-эффект event-study DiD (п.п.) с 95% CI",
        ylabel="Эффект, п.п.",
    )
    # individual stems for thesis figure names when weekly missing
    for outcome in OUTCOMES_MAIN:
        plot_post_bar_chart(
            post_summary,
            outcomes=[outcome],
            estimate_col=est,
            lo_col=lo,
            hi_col=hi,
            stem=STEM_MAP[outcome].replace("event_study_", "post_effect_"),
            title=f"Post-эффект: {OUTCOME_LABELS[outcome]}",
            ylabel="Эффект, п.п.",
        )

    if not speed_summary.empty:
        sp = speed_summary.copy()
        est_col = next(
            (c for c in ["avg_post_effect_days", "avg_weighted", "avg_post_effect_days_weighted"] if c in sp.columns),
            None,
        )
        if est_col:
            sp["outcome"] = "t_available"
            plot_post_bar_chart(
                sp,
                outcomes=["t_available"],
                estimate_col=est_col,
                lo_col=None,
                hi_col=None,
                stem="post_effect_t_available_by_type",
                title="Средний post-эффект: t_available",
                ylabel="Дни",
            )

saved: figures\empirical\event_study_sch_by_type.pdf


saved: figures\empirical\event_study_meet_by_type.pdf


saved: figures\empirical\event_study_success_by_type.pdf


saved: figures\empirical\event_study_utlz_by_type.pdf


saved: figures\empirical\event_study_t_available_by_type.pdf


## ATT(g,t) / differences (если CSV доступны)

Дополнительные столбчатые диаграммы средних post-ATT из `differences_post_average_summary.csv` и event-style графики из `differences_event_summary.csv`, без переоценки.

In [7]:
if not diff_post.empty:
    dp = diff_post.copy()
    if "outcome" in dp.columns:
        dp["outcome"] = dp["outcome"].map(canonicalize_outcome)
    att_col = next(
        (c for c in ["ATT_pp", "estimate_pp", "estimate"] if c in dp.columns),
        None,
    )
    lo_col = next((c for c in ["lower_pp", "ci_lower_pp", "ci_lower"] if c in dp.columns), None)
    hi_col = next((c for c in ["upper_pp", "ci_upper_pp", "ci_upper"] if c in dp.columns), None)
    if att_col:
        conv = [o for o in OUTCOMES_MAIN if "outcome" in dp.columns and (dp["outcome"] == o).any()]
        if conv:
            plot_post_bar_chart(
                dp,
                outcomes=conv,
                estimate_col=att_col,
                lo_col=lo_col,
                hi_col=hi_col,
                stem="differences_post_att_conversions",
                title="ATT(g,t) post-average (differences)",
                ylabel="ATT, п.п.",
            )
else:
    print("differences_post_average_summary.csv не найден — пропускаем ATT bars.")

if not diff_event.empty and has_weekly_coefs(
    diff_event.rename(columns={
        "relative_period": "event_week",
        "estimate": "coef",
        "ci_lower": "ci_low",
        "ci_upper": "ci_high",
    }) if {"estimate", "ci_lower", "ci_upper"}.issubset(diff_event.columns) else pd.DataFrame()
):
    de = diff_event.copy()
    if "aggregation" in de.columns:
        de = de[de["aggregation"].astype(str).str.lower().eq("event")].copy()
    if "relative_period" in de.columns and "event_week" not in de.columns:
        de["event_week"] = np.floor(pd.to_numeric(de["relative_period"], errors="coerce") / 7)
    de = de.rename(columns={"estimate": "coef", "ci_lower": "ci_low", "ci_upper": "ci_high"})
    if "outcome" in de.columns:
        de["outcome"] = de["outcome"].map(canonicalize_outcome)
        for outcome in OUTCOMES_MAIN:
            if (de["outcome"] == outcome).any():
                plot_event_study_from_coefs(
                    outcome, de, f"differences_attgt_weekly_{outcome}"
                )
else:
    print("Недельные differences event-coefs недоступны или без CI — пропускаем.")

saved: figures\empirical\differences_post_att_conversions.pdf


saved: figures\empirical\differences_attgt_weekly_sch_flg.pdf


saved: figures\empirical\differences_attgt_weekly_meet_flg.pdf


saved: figures\empirical\differences_attgt_weekly_success_flg.pdf


saved: figures\empirical\differences_attgt_weekly_utlz_within_25.pdf


In [8]:
# Канонический финальный проход: все рисунки, подключённые в thesis/main.tex
from last_mile.publication_figures import generate_all

publication_paths = generate_all(PROJECT_ROOT)
for publication_path in publication_paths:
    print("готово для публикации:", publication_path.relative_to(PROJECT_ROOT))


готово для публикации: figures\empirical\cohort_composition_core_shares.pdf
готово для публикации: figures\empirical\pretrends_sch_by_cohort.pdf
готово для публикации: figures\empirical\pretrends_success_by_cohort.pdf
готово для публикации: figures\empirical\event_study_t_available_by_type.pdf
готово для публикации: figures\empirical\event_study_sch_by_type.pdf
готово для публикации: figures\empirical\event_study_meet_by_type.pdf
готово для публикации: figures\empirical\event_study_success_by_type.pdf
готово для публикации: figures\empirical\event_study_utlz_by_type.pdf
готово для публикации: figures\empirical\threshold_tradeoff_publication.pdf
готово для публикации: figures\empirical\did_threshold_sensitivity_natural_vs_common.pdf


## Вывод

Ноутбук воспроизводит финальные рисунки из канонических CSV без изменения оценок. Графики событийной модели и постэффектов согласованы с `main_event_study_post_summary.csv`; итоговые PDF используются в `thesis/main.tex`.